In [ ]:
import pandas as pd, numpy as np
from google.colab import files

files.upload()          # choose healthcare_dataset.csv
df = pd.read_csv('healthcare_dataset.csv')
print(df.shape)
df.head()


Saving healthcare_dataset.csv to healthcare_dataset.csv
(55500, 15)


,Name,Age,Gender,Blood Type,Medical Condition,Date of Admission,Doctor,Hospital,Insurance Provider,Billing Amount,Room Number,Admission Type,Discharge Date,Medication,Test Results
0,Bobby JacksOn,30,Male,B-,Cancer,2024-01-31,Matthew Smith,Sons and Miller,Blue Cross,18856.281306,328,Urgent,2024-02-02,Paracetamol,Normal
1,LesLie TErRy,62,Male,A+,Obesity,2019-08-20,Samantha Davies,Kim Inc,Medicare,33643.327287,265,Emergency,2019-08-26,Ibuprofen,Inconclusive
2,DaNnY sMitH,76,Female,A-,Obesity,2022-09-22,Tiffany Mitchell,Cook PLC,Aetna,27955.096079,205,Emergency,2022-10-07,Aspirin,Normal
3,andrEw waTtS,28,Female,O+,Diabetes,2020-11-18,Kevin Wells,"Hernandez Rogers and Vang,",Medicare,37909.782410,450,Elective,2020-12-18,Ibuprofen,Abnormal
4,adrIENNE bEll,43,Female,AB+,Cancer,2022-09-19,Kathleen Hanna,White-White,Aetna,14238.317814,458,Urgent,2022-10-09,Penicillin,Abnormal


In [ ]:
df.info()
print(df.isna().sum())
print('Duplicates:', df.duplicated().sum())
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 55500 entries, 0 to 55499
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Name                55500 non-null  object 
 1   Age                 55500 non-null  int64  
 2   Gender              55500 non-null  object 
 3   Blood Type          55500 non-null  object 
 4   Medical Condition   55500 non-null  object 
 5   Date of Admission   55500 non-null  object 
 6   Doctor              55500 non-null  object 
 7   Hospital            55500 non-null  object 
 8   Insurance Provider  55500 non-null  object 
 9   Billing Amount      55500 non-null  float64
 10  Room Number         55500 non-null  int64  
 11  Admission Type      55500 non-null  object 
 12  Discharge Date      55500 non-null  object 
 13  Medication          55500 non-null  object 
 14  Test Results        55500 non-null  object 
dtypes: float64(1), int64(2), object(12)
memory usage: 6.4

,Age,Billing Amount,Room Number
count,55500.000000,55500.000000,55500.000000
mean,51.539459,25539.316097,301.134829
std,19.602454,14211.454431,115.243069
min,13.000000,-2008.492140,101.000000
25%,35.000000,13241.224652,202.000000
50%,52.000000,25538.069376,302.000000
75%,68.000000,37820.508436,401.000000
max,89.000000,52764.276736,500.000000


In [ ]:
print('Negative billing rows:', (df['Billing Amount'] < 0).sum())
print('Unique hospitals:', df['Hospital'].nunique())
print('Unique doctors  :', df['Doctor'].nunique())
print(df['Medical Condition'].value_counts())
print(df['Admission Type'].value_counts())

Negative billing rows: 108
Unique hospitals: 39876
Unique doctors  : 40341
Medical Condition
Arthritis       9308
Diabetes        9304
Hypertension    9245
Obesity         9231
Cancer          9227
Asthma          9185
Name: count, dtype: int64
Admission Type
Elective     18655
Urgent       18576
Emergency    18269
Name: count, dtype: int64


In [ ]:
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
df['name'] = df['name'].str.title()
rows_before = len(df)
df = df.drop_duplicates()
print('Duplicates removed:', rows_before - len(df))

Duplicates removed: 534


In [ ]:
df['date_of_admission'] = pd.to_datetime(df['date_of_admission'])
df['discharge_date']    = pd.to_datetime(df['discharge_date'])
df['billing_amount']    = df['billing_amount'].round(2)

In [ ]:
df['billing_flag'] = np.where(df['billing_amount'] < 0, 'Invalid', 'Valid')
print(df['billing_flag'].value_counts())

billing_flag
Valid      54860
Invalid      106
Name: count, dtype: int64


/tmp/ipykernel_861/4281487197.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['billing_flag'] = np.where(df['billing_amount'] < 0, 'Invalid', 'Valid')


In [ ]:
df['length_of_stay'] = (df['discharge_date'] - df['date_of_admission']).dt.days
df['age_group'] = pd.cut(df['age'], [0,17,35,50,65,120],
                         labels=['0-17','18-35','36-50','51-65','66+'])
df['admit_month'] = df['date_of_admission'].dt.to_period('M').astype(str)
print('Negative stays:', (df['length_of_stay'] < 0).sum())
print(df['length_of_stay'].describe())

Negative stays: 0
count    54966.000000
mean        15.499290
std          8.661471
min          1.000000
25%          8.000000
50%         15.000000
75%         23.000000
max         30.000000
Name: length_of_stay, dtype: float64


## **Q1- most common conditions.**

In [ ]:
q1 = df['medical_condition'].value_counts()
print(q1)

medical_condition
Arthritis       9218
Diabetes        9216
Hypertension    9151
Obesity         9146
Cancer          9140
Asthma          9095
Name: count, dtype: int64


# **Q2-longest average stay.**

In [ ]:
q2 = df.groupby('medical_condition')['length_of_stay'].mean().round(1).sort_values(ascending=False)
print(q2)

medical_condition
Asthma          15.7
Arthritis       15.5
Cancer          15.5
Diabetes        15.4
Hypertension    15.4
Obesity         15.4
Name: length_of_stay, dtype: float64


## **Q3-insurance billing (valid rows only).**

In [ ]:
valid = df[df['billing_flag'] == 'Valid']
q4 = valid.groupby('insurance_provider')['billing_amount'].agg(['count','sum','mean']).round(0).sort_values('sum', ascending=False)
print(q4)

KeyError: 'billing_flag'

# ** Q4-admission mix.**

In [ ]:
top = df['hospital'].value_counts().head(10).index
q3 = pd.crosstab(df[df['hospital'].isin(top)]['hospital'], df['admission_type'], normalize='index').round(3)
print(q3)

admission_type  Elective  Emergency  Urgent
hospital                                   
Group Smith        0.375      0.344   0.281
Johnson Inc        0.471      0.294   0.235
Johnson PLC        0.324      0.324   0.351
LLC Smith          0.227      0.432   0.341
Ltd Smith          0.308      0.282   0.410
Smith Group        0.361      0.278   0.361
Smith Inc          0.333      0.242   0.424
Smith LLC          0.469      0.281   0.250
Smith Ltd          0.351      0.378   0.270
Smith PLC          0.389      0.333   0.278


# **Q5-monthly trend.**

In [ ]:
q5 = df.groupby('admit_month').size()
print(q5)

admit_month
2019-05    677
2019-06    899
2019-07    951
2019-08    985
2019-09    924
          ... 
2024-01    903
2024-02    869
2024-03    902
2024-04    941
2024-05    212
Length: 61, dtype: int64


# **Q6-medication patterns.**

In [ ]:
q6a = pd.crosstab(df['medication'], df['medical_condition'])
q6b = pd.crosstab(df['medication'], df['test_results'], normalize='index').round(3)
print(q6a); print(q6b)

medical_condition  Arthritis  Asthma  Cancer  Diabetes  Hypertension  Obesity
medication                                                                   
Aspirin                 1901    1781    1768      1836          1845     1853
Ibuprofen               1805    1802    1862      1846          1874     1834
Lipitor                 1810    1814    1904      1875          1823     1812
Paracetamol             1858    1870    1829      1794          1839     1775
Penicillin              1844    1828    1777      1865          1770     1872
test_results  Abnormal  Inconclusive  Normal
medication                                  
Aspirin          0.337         0.327   0.336
Ibuprofen        0.337         0.328   0.336
Lipitor          0.332         0.337   0.330
Paracetamol      0.337         0.333   0.331
Penicillin       0.334         0.331   0.334


In [ ]:
print('Final rows      :', len(df))
print('Invalid billing :', (df['billing_flag']=='Invalid').sum())
print('Date range      :', df['date_of_admission'].min().date(), 'to', df['date_of_admission'].max().date())

Final rows      : 54966
Invalid billing : 106
Date range      : 2019-05-08 to 2024-05-07


In [ ]:
df.to_csv('admissions_clean.csv', index=False)
files.download('admissions_clean.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
print(df.columns.tolist())
print(df.columns[df.columns.duplicated()])

['name', 'age', 'gender', 'blood_type', 'medical_condition', 'date_of_admission', 'doctor', 'hospital', 'insurance_provider', 'billing_amount', 'room_number', 'admission_type', 'discharge_date', 'medication', 'test_results', 'length_of_stay', 'age_group', 'admit_month']
Index([], dtype='object')
